# Tune the position filter

Sweep the parameters of a position filter for a visual-proprioception run and
report the per-field optimum.

The run given below is a **filtered variant** (one declaring `base_run`); this
notebook resolves its base run for the sensor processor, the trained regressor
and the cached latents. It sweeps on the run's `training_data` and never reads
`validation_data`, so the values it reports are not tuned against the set the
comparison notebooks report on.

Nothing is saved. Paste the printed block into the variant's exp/run yaml.

See `DESIGN-TemporalVisualProprioception.md`.


In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt

from visual_proprioception.visproprio_helper import (
    predict_positions, resolve_base_experiment)
from visual_proprioception.visproprio_filters import (
    estimate_noise, tune_ema, tune_kalman)

FIELDS = ["height", "distance", "heading",
          "wrist_angle", "wrist_rotation", "gripper"]

device = Config().runtime["device"]
print(f"Using device: {device}")


In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

experiment = "visual_proprioception"

# A filtered variant; its base run supplies the model and the latents.
run = "vp_ptun_vgg19_128_ema"
# run = "vp_ptun_vgg19_128_kalman"
# run = "vp_convvae_128_ema"
# run = "vp_convvae_128_kalman"

# If not None, set an external experiment path
expruns_path = None

# If not None, set an output path
results_path = None


In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path).expanduser()
    assert expruns_path.exists()
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment("visual_proprioception")
    Config().copy_experiment("sensorprocessing_conv_vae")
    Config().copy_experiment("sensorprocessing_propriotuned_cnn")
    Config().copy_experiment("robot_al5d")
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path).expanduser()
    assert results_path.exists()
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run)
pprint(exp)

exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])
base = resolve_base_experiment(exp)
print(f"\nTuning '{exp['position_filter']}' for {exp['name']}, "
      f"using the model of {base['name']}")


## Unfiltered predictions on the training demonstrations

The sweep needs the regressor's raw output, so the filter configured by the
run is bypassed here: only `predictions` is used below, never `filtered`.


In [ ]:
result = predict_positions(exp, exp_robot, "training_data")

targets = result["targets"]
predictions = result["predictions"]
lengths = result["lengths"]
dt = exp["sample_interval"]

print(f"{len(predictions)} training frames in {len(lengths)} demonstrations: {lengths}")
print(f"observation interval {dt} s\n")
print(f"{'field':<16}{'unfiltered RMSE':>18}{'mean motion/frame':>20}")
for index, field in enumerate(FIELDS):
    rmse = np.sqrt(np.mean((predictions[:, index] - targets[:, index]) ** 2))
    motion = np.abs(np.diff(targets[:, index])).mean()
    print(f"{field:<16}{rmse:18.4f}{motion:20.4f}")


## Noise terms and sweep

Each field is tuned independently, because the filters treat the degrees of freedom independently. The EMA sweep is over the time constant; the Kalman sweep is over a multiplier on the measured measurement noise, which is the ratio the estimator is actually sensitive to.


In [ ]:
# Starting noise terms measured from this training set: measurement noise is
# the variance of the regressor's residual, process noise the acceleration
# noise density of the commanded trajectory within each demonstration.
process_noise, measurement_noise = estimate_noise(
    predictions, targets, lengths, dt)

print(f"{'field':<16}{'process noise':>16}{'measurement noise':>20}")
for index, field in enumerate(FIELDS):
    print(f"{field:<16}{process_noise[index]:16.5f}"
          f"{measurement_noise[index]:20.5f}")


In [ ]:
if exp["position_filter"] == "ema":
    tuned = tune_ema(predictions, targets, lengths, dt)
    label = "tau (s)"
elif exp["position_filter"] == "kalman":
    tuned = tune_kalman(predictions, targets, lengths, dt,
                        process_noise, measurement_noise)
    label = "measurement noise multiplier"
else:
    raise Exception(
        f"Nothing to tune for position_filter '{exp['position_filter']}'")

unfiltered = np.sqrt(np.mean((predictions - targets) ** 2, axis=0))

print(f"{'field':<16}{'unfiltered':>12}{'best':>10}{'gain':>8}{label:>32}")
for index, field in enumerate(FIELDS):
    print(f"{field:<16}{unfiltered[index]:12.4f}{tuned['rmse'][index]:10.4f}"
          f"{unfiltered[index] / tuned['rmse'][index]:7.2f}x"
          f"{tuned['parameters'][index]:32.4f}")


## Sweep curves

A flat minimum means the parameter is not critical; a sharp one means the run
is sensitive to it. A curve whose minimum sits at the edge of the sweep means
the range needs widening.


In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(11, 6), constrained_layout=True)
for index, field in enumerate(FIELDS):
    ax = axs[index // 3, index % 3]
    ax.plot(tuned["candidates"], tuned["errors"][:, index], label="filtered")
    ax.axhline(unfiltered[index], color="black", linestyle="--",
               linewidth=1, label="unfiltered")
    ax.axvline(tuned["parameters"][index], color="red",
               linewidth=1, label="best")
    ax.set_xscale("log")
    ax.set_xlabel(label)
    ax.set_ylabel("RMSE")
    ax.set_title(field)
    if index == 0:
        ax.legend()

graphfilename = pathlib.Path(exp["data_dir"], "filter_tuning.pdf")
plt.savefig(graphfilename, bbox_inches="tight")
print(f"Saved {graphfilename}")


## Values for the exp/run

Paste this into the variant's yaml. The comparison notebooks then report the
filtered result with these settings.


In [ ]:
print(f"# tuned on {len(predictions)} training frames of {base['name']}")
print(f"position_filter: {exp['position_filter']}")
if exp["position_filter"] == "ema":
    print("filter_tau: ["
          + ", ".join(f"{value:.3f}" for value in tuned["parameters"]) + "]")
else:
    print("filter_process_noise: ["
          + ", ".join(f"{value:.5f}" for value in process_noise) + "]")
    print("filter_measurement_noise: ["
          + ", ".join(f"{value:.5f}"
                      for value in measurement_noise * tuned["parameters"])
          + "]")
